# 02 — 特征工程
**任务：多景拼接 → 最大NDVI月合成 → 特征矩阵输出**

测试区：悉尼西部农业区（147°E–149°E，34°S–32.5°S）

时间：**2024年4月**（与侵蚀标签时间完全匹配，用于模型训练）

输出：特征矩阵 `NSW_test_features_2024_04.npy`（H × W × 7通道）
- 通道顺序：NDVI, NDWI, BSI, SAVI, B04, B08, B11

## 1. 导入包

In [7]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pystac_client import Client
import planetary_computer as pc
import stackstac
import rasterio
import gc
import subprocess
from tqdm import tqdm

from src.data_utils import load_config, ensure_dir
from src.indices import calculate_all_indices

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('所有包导入成功')

所有包导入成功


## 2. 加载配置

In [8]:
cfg = load_config('../configs/config.yaml')

bbox          = cfg['region']['bbox_test']
BBOX          = [bbox['lon_min'], bbox['lat_min'], bbox['lon_max'], bbox['lat_max']]
EPSG          = 32755
CLOUD_MAX     = 5
STAC_URL      = cfg['sentinel2']['stac_endpoint']
TIME_RANGE    = '2024-04-01/2024-05-01'   # 与标签一致
LABEL         = '2024_04'

processed_dir = ensure_dir('../data/processed')
fig_dir       = ensure_dir('../results/figures')

print(f'研究区 BBOX : {BBOX}')
print(f'时间范围    : {TIME_RANGE}')
print(f'最大云量    : {CLOUD_MAX}%')
print(f'标签        : {LABEL}')

研究区 BBOX : [147.0, -34.0, 149.0, -32.5]
时间范围    : 2024-04-01/2024-05-01
最大云量    : 5%
标签        : 2024_04


## 3. 搜索 2024年4月场景

In [9]:
catalog   = Client.open(STAC_URL)
search    = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': CLOUD_MAX}}
)
raw_items = list(search.items())
print(f'找到场景数: {len(raw_items)}')

# 按日期分组
raw_dates = {}
for item in raw_items:
    date = item.datetime.strftime('%Y-%m-%d')
    if date not in raw_dates:
        raw_dates[date] = []
    raw_dates[date].append(item)

print(f'共 {len(raw_dates)} 个日期:')
for date, sitems in sorted(raw_dates.items()):
    print(f'  {date}: {len(sitems)} 景')

找到场景数: 56
共 11 个日期:
  2024-04-01: 8 景
  2024-04-03: 2 景
  2024-04-11: 9 景
  2024-04-13: 3 景
  2024-04-14: 4 景
  2024-04-16: 9 景
  2024-04-18: 1 景
  2024-04-21: 7 景
  2024-04-23: 3 景
  2024-04-26: 9 景
  2024-05-01: 1 景


## 4. 选取景数最多的3个日期

In [10]:
TOP_N      = 3
best_dates = sorted(raw_dates.items(), key=lambda x: len(x[1]), reverse=True)[:TOP_N]

print(f'选取覆盖率最高的 {TOP_N} 个日期:')
for date, sitems in best_dates:
    print(f'  {date}: {len(sitems)} 景')

选取覆盖率最高的 3 个日期:
  2024-04-26: 9 景
  2024-04-16: 9 景
  2024-04-11: 9 景


## 5. 分批下载并计算最大NDVI合成

In [ ]:
daily_ndvi    = []
daily_bands   = []
transform_ref = None
height = width = None
gc.collect()

for date, raw_scene_items in tqdm(best_dates, desc='按日期处理'):
    try:
        # 每次循环即时签名，防止过期
        scene_items = [pc.sign(item) for item in raw_scene_items]
        day_stack   = stackstac.stack(
            scene_items,
            assets=['B02','B03','B04','B08','B11'],
            bounds_latlon=BBOX,
            resolution=10,
            dtype='float64',
            rescale=False,
            epsg=EPSG
        )
        day_data = day_stack.compute()

        b04   = day_data.sel(band='B04').values / 10000.0
        b08   = day_data.sel(band='B08').values / 10000.0
        valid = (b04 > 0) & (b08 > 0) & np.isfinite(b04) & np.isfinite(b08)
        b04[~valid] = np.nan
        b08[~valid] = np.nan

        ndvi_day  = (b08 - b04) / (b08 + b04 + 1e-8)
        ndvi_day[~valid] = np.nan

        has_valid = valid.any(axis=0)
        ndvi_max  = np.where(has_valid,
                             np.nanmax(np.where(valid, ndvi_day, -999), axis=0),
                             np.nan)
        best_t    = np.where(has_valid,
                             np.nanargmax(np.where(valid, ndvi_day, -999), axis=0),
                             0).astype(int)
        rows, cols = np.indices(best_t.shape)

        b02 = day_data.sel(band='B02').values / 10000.0
        b03 = day_data.sel(band='B03').values / 10000.0
        b11 = day_data.sel(band='B11').values / 10000.0
        for b in [b02, b03, b04, b08, b11]:
            b[~valid] = np.nan

        daily_ndvi.append(ndvi_max)
        daily_bands.append([
            b02[best_t, rows, cols],
            b03[best_t, rows, cols],
            b04[best_t, rows, cols],
            b08[best_t, rows, cols],
            b11[best_t, rows, cols],
        ])

        if transform_ref is None:
            transform_ref = day_stack.transform
            height, width = ndvi_max.shape

        del day_data, day_stack
        gc.collect()
        print(f'  {date} 完成，有效像素: {np.isfinite(ndvi_max).mean()*100:.1f}%')

    except Exception as e:
        print(f'  {date} 跳过: {e}')

print(f'\n共处理 {len(daily_ndvi)} 个日期')

按日期处理:  33%|███▎      | 1/3 [17:55<35:50, 1075.44s/it]

  2024-04-26 完成，有效像素: 100.0%


## 6. 跨日期最大NDVI合成

In [ ]:
ndvi_stack = np.stack(daily_ndvi, axis=0)
has_valid  = np.isfinite(ndvi_stack).any(axis=0)
final_ndvi = np.where(has_valid, np.nanmax(ndvi_stack, axis=0), np.nan)
best_day   = np.where(has_valid,
                      np.nanargmax(np.where(np.isfinite(ndvi_stack), ndvi_stack, -999), axis=0),
                      0).astype(int)
rows, cols = np.indices(best_day.shape)

B02 = np.stack([b[0] for b in daily_bands], axis=0)[best_day, rows, cols]
B03 = np.stack([b[1] for b in daily_bands], axis=0)[best_day, rows, cols]
B04 = np.stack([b[2] for b in daily_bands], axis=0)[best_day, rows, cols]
B08 = np.stack([b[3] for b in daily_bands], axis=0)[best_day, rows, cols]
B11 = np.stack([b[4] for b in daily_bands], axis=0)[best_day, rows, cols]

valid_pct = np.isfinite(final_ndvi).mean() * 100
print(f'月合成完成')
print(f'形状      : {final_ndvi.shape}')
print(f'有效像素  : {valid_pct:.1f}%')
print(f'NDVI 范围 : {np.nanmin(final_ndvi):.3f} ~ {np.nanmax(final_ndvi):.3f}')

## 7. 计算遥感指数

In [ ]:
indices = calculate_all_indices(
    {'blue': B02, 'green': B03, 'red': B04, 'nir': B08, 'swir': B11}
)

print(f'{"指数":<6} {"最小值":>8} {"最大值":>8} {"均值":>8} {"有效像素%":>10}')
print('-' * 46)
for name, arr in indices.items():
    v   = arr[np.isfinite(arr)]
    pct = len(v) / arr.size * 100
    print(f'{name:<6} {v.min():>8.3f} {v.max():>8.3f} {v.mean():>8.3f} {pct:>10.1f}%')

## 8. 可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NSW 测试区 Sentinel-2 月合成指数\n2024年4月（最大NDVI合成）', fontsize=14)

plots = [
    ('NDVI', indices['NDVI'], 'RdYlGn', -0.2, 0.8),
    ('NDWI', indices['NDWI'], 'Blues',  -0.4, 0.4),
    ('BSI',  indices['BSI'],  'YlOrBr', -0.3, 0.3),
    ('SAVI', indices['SAVI'], 'RdYlGn', -0.2, 0.8),
]

for ax, (name, arr, cmap, vmin, vmax) in zip(axes.flat, plots):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
fig_path = fig_dir / f'NSW_composite_indices_{LABEL}.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'图像已保存: {fig_path.name}')
plt.show()

## 9. 保存特征矩阵

In [ ]:
# 保存各指数 GeoTIFF
for name, arr in indices.items():
    out_path = processed_dir / f'NSW_test_{name}_composite_{LABEL}.tif'
    with rasterio.open(
        out_path, 'w', driver='GTiff',
        height=height, width=width, count=1,
        dtype='float32', crs=f'EPSG:{EPSG}',
        transform=transform_ref,
        compress='lzw', nodata=np.nan
    ) as dst:
        dst.write(arr.astype('float32'), 1)
    print(f'已保存: {out_path.name}')

# 保存特征矩阵 (H, W, 7)
feature_stack = np.stack([
    indices['NDVI'], indices['NDWI'],
    indices['BSI'],  indices['SAVI'],
    B04, B08, B11
], axis=-1).astype('float32')

npy_path = processed_dir / f'NSW_test_features_{LABEL}.npy'
np.save(npy_path, feature_stack)

print(f'\n特征矩阵已保存: {npy_path.name}')
print(f'形状          : {feature_stack.shape}  (H × W × 7通道)')
print(f'7通道依次为   : NDVI, NDWI, BSI, SAVI, B04, B08, B11')

## 10. 提交到 GitHub

In [ ]:
commands = [
    ['git', 'add', 'notebooks/02_feature_engineering.ipynb'],
    ['git', 'add', f'results/figures/NSW_composite_indices_{LABEL}.png'],
    ['git', 'commit', '-m', f'feat: notebook 02 - 2024-04 Sentinel-2 features aligned with erosion label'],
    ['git', 'push']
]

for cmd in commands:
    result = subprocess.run(cmd, cwd='..', capture_output=True, text=True)
    print(' '.join(cmd))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)